# ResNet-34 под cifar 10

In [1]:
import torch
import torchvision

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import random_split, TensorDataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Normalize, RandomCrop, RandomHorizontalFlip

In [2]:
class BasicBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()

        self.layer = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            torch.nn.BatchNorm2d(out_channels),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, stride=1, padding=padding),
            torch.nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        return self.layer(x)

In [3]:
class FullBlock(torch.nn.Module):
    def __init__(self, count_blocks, input_channels, output_channels, kernel_size, stride, padding):
        super().__init__()
        self.count_blocks = count_blocks

        self.change_block = BasicBlock(input_channels, output_channels, kernel_size, stride, padding)
        self.blocks = torch.nn.Sequential()
        self._constract_blocks(output_channels, kernel_size)

        self.act = torch.nn.ReLU()
        self.connection = torch.nn.Conv2d(input_channels, output_channels, kernel_size=1, stride=stride)

    def _constract_blocks(self, channels, kernel_size):
        for i in range(1, self.count_blocks):
            self.blocks.add_module(f"block_{i+1}", BasicBlock(in_channels=channels,
                                                                out_channels=channels, 
                                                                kernel_size=kernel_size,
                                                                stride=1,
                                                                padding=1,
                                                            ))

    def forward(self, x):
        out = self.act(self.connection(x) + self.change_block(x))

        for block in self.blocks:
            out = self.act(out + block(out))

        return out

class Stem(torch.nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size, stride, padding):
        super().__init__()
        self.layer = torch.nn.Sequential(
            torch.nn.Conv2d(input_channels, output_channels, kernel_size, stride, padding),
            torch.nn.BatchNorm2d(output_channels),
            torch.nn.ReLU(),
        )

    def forward(self, x):
        return self.layer(x)

    

In [4]:
class ResNet34(torch.nn.Module):
    def __init__(self, input_channels: int=3, n_output: int=10):
        super().__init__()

        self.layers = torch.nn.Sequential(
            Stem(input_channels=input_channels, output_channels=64, kernel_size=3, stride=1, padding=1),

            FullBlock(count_blocks=3, input_channels=64, output_channels=64, kernel_size=3, stride=1, padding=1),   # 32 -> 32
            FullBlock(count_blocks=4, input_channels=64, output_channels=128, kernel_size=3, stride=2, padding=1),  # 32 -> 16
            FullBlock(count_blocks=6, input_channels=128, output_channels=256, kernel_size=3, stride=2, padding=1), # 16 -> 8
            FullBlock(count_blocks=3, input_channels=256, output_channels=512, kernel_size=3, stride=2, padding=1), # 8 -> 4

            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),

            torch.nn.Linear(in_features=512, out_features=64),
            torch.nn.BatchNorm1d(64),
            torch.nn.Dropout(),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features=64, out_features=n_output)
        )

    def forward(self, x):
        return self.layers(x)       

In [5]:
dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=ToTensor()
)

mean_r = (dataset.data[...,0].astype("float32") /255).mean()
mean_g = (dataset.data[...,1].astype("float32") /255).mean()
mean_b = (dataset.data[...,2].astype("float32") /255).mean()

std_r = (dataset.data[...,0].astype("float32") /255).std()
std_g = (dataset.data[...,1].astype("float32") /255).std()
std_b = (dataset.data[...,2].astype("float32") /255).std()

In [6]:
mean_r, mean_g, mean_b

(np.float32(0.49139968), np.float32(0.48215845), np.float32(0.4465309))

In [9]:
transform1 = torchvision.transforms.Compose([
    ToTensor(),
    Normalize((mean_r, mean_g, mean_b), (std_r, std_g, std_b)),
])

transform2 = torchvision.transforms.Compose([
    RandomCrop(32, padding=4),
    RandomHorizontalFlip(),
    ToTensor(),
    Normalize((mean_r, mean_g, mean_b), (std_r, std_g, std_b)),
])

In [10]:
def data_load(transform, train: bool=True):
    dataset = datasets.CIFAR10(
        root="./data",
        train=train,
        download=True,
        transform=transform,
    )
    return dataset

train_dataset = data_load(transform=transform2, train=True)
t_v_dataset = data_load(transform=transform1, train=False)

In [11]:
val_size = int(0.5 * len(t_v_dataset))
test_size = len(t_v_dataset) - val_size

val_dataset, test_dataset = random_split(t_v_dataset, lengths=(val_size, test_size))

In [12]:
model = ResNet34()
batch_size = 128
EPOCHS = 200

In [14]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

criterion = torch.nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=5)

In [15]:
loop = tqdm(range(EPOCHS), desc="Training ResNet34")
train_acc_history = []
train_loss_history = []
val_acc_history = []
val_loss_history = []
lr_history = []
pred_loss_train = float("inf")
pred_loss_val = float("inf")

for epoch in loop:

    total = 0
    correct = 0
    losses = []

    model.train()
    loop_train = tqdm(enumerate(train_loader), desc=f"ResNet34 on training [{epoch+1}/{EPOCHS}]", total=len(train_loader), leave=False)
    for i, (images, targets) in loop_train:
        optim.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)

        losses.append(loss.item())

        temp_total = len(targets)
        total += temp_total
        temp_correct = torch.sum(torch.argmax(logits, axis=1) == targets).item()
        correct += temp_correct

        loss.backward()
        optim.step()

        if i % 10 == 0:
            loop_train.set_postfix(loss=loss.item(), accuracy=temp_correct/temp_total)

    train_acc_history.append(correct/total)
    train_loss_history.append(np.mean(losses))
    loop.set_postfix(mean_train_loss=train_loss_history[-1], mean_val_loss=pred_loss_val, lr=optim.param_groups[0]["lr"])
    pred_loss_train = train_loss_history[-1]

    total = 0
    correct = 0
    losses = []

    model.eval()
    loop_val = tqdm(enumerate(val_loader), desc=f"ResNet34 on validation [{epoch+1}/{EPOCHS}]", total=len(val_loader), leave=False)
    for i, (images, targets) in loop_val:
        with torch.no_grad():
            logits = model(images)
            loss = criterion(logits, targets)

            losses.append(loss.item())

            temp_total = len(targets)
            total += temp_total
            temp_correct = torch.sum(torch.argmax(logits, axis=1) == targets).item()
            correct += temp_correct

            if i % 10 == 0:
                loop_val.set_postfix(loss=loss.item(), accuracy=temp_correct/temp_total)

    val_acc_history.append(correct/total)
    val_loss_history.append(np.mean(losses))
    loop.set_postfix(mean_train_loss=pred_loss_train, mean_val_loss=val_loss_history[-1], lr=optim.param_groups[0]["lr"])
    pred_loss_val = val_loss_history[-1]

    sheduler.step(val_acc_history[-1])

    lr_history.append(lr=optim.param_groups[0]["lr"])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8, 4))

ax1.plot(train_loss_history, label="train")
ax1.plot(val_loss_history, label="validation")
ax1.set_title("Loss history")
ax1.legend()

ax2.plot(train_acc_history, label="train")
ax2.plot(val_acc_history, label="validation")
ax2.set_title("Accuracy history")
ax2.legend()

ax3.plot(train_acc_history)
ax2.set_title("Learning rate history")
ax2.legend()


Training ResNet34:   0%|          | 0/200 [00:00<?, ?it/s]

ResNet34 on training [1/200]:   0%|          | 0/391 [00:00<?, ?it/s]

KeyboardInterrupt: 